In [ ]:
#Import libraries
import tensorflow.keras
from numpy import ndarray
import skimage.io as io
import skimage.transform as trans
from tensorflow.nn import *
from keras.models import *
from keras.layers import *
from keras.optimizers import *
from keras.losses import *
from keras.callbacks import ModelCheckpoint, LearningRateScheduler
from keras import backend as keras
from keras.preprocessing.image import ImageDataGenerator
#from keras.layers import Conv2D, BatchNormalization, MaxPooling2D, Dropout, Dense, Conv2DTranspose
from tensorflow.keras import Sequential
import pandas as pd
# labels = pd.read_csv('AlzheimersFLAIR/patient_labels.csv')
# patient_labels.csv is not included in this repository due to ADNI data-use restrictions.

In [ ]:
def contrast_stretch(image, new_min=0, new_max=1):
    # Compute the current min and max intensity values
    current_min = np.min(image)
    current_max = np.max(image)
    
    # Apply contrast stretching
    stretched_image = (image - current_min) * (new_max - new_min) / (current_max - current_min) + new_min
    
    return stretched_image

In [ ]:
import nibabel as nib
import numpy as np
import os
from scipy.ndimage import gaussian_filter
import cv2
from PIL import Image

directory = 'segmented-images'
volume_directory = 'AlzheimersFLAIR/data-preprocessing/n4-corrected2'

mask_array = []
volume_array = []
min_array = []
max_array = []

total_images = 0

used_patients = []

for s in os.listdir(directory):
    if (s != '.DS_Store'):
        filepath = os.path.join(directory, s)
        target_size = (224, 224)
        
        img = contrast_stretch(nib.load(filepath).get_fdata().transpose(2, 1, 0))
        #print(np.shape(img))
        min_i = 0
        max_i = 0
        
        volume_path = os.path.join(volume_directory, s[4:] + '.gz')
        volume = np.array([contrast_stretch(i) * 255 for i in nib.load(volume_path).get_fdata().transpose(2, 1, 0)]).astype(np.uint8)
        masks = []
        images = []
        for i in range(len(img)):
            segmented = np.sum(img[i]) > 0
            if (segmented):
                if (min_i == 0):
                    min_i = i
                max_i = i
                masks.append(cv2.resize(img[i], target_size, interpolation = cv2.INTER_LINEAR))
                images.append(cv2.resize(volume[i], target_size, interpolation = cv2.INTER_LINEAR))
                #print(i, np.sum(img[i]))
        if (max_i - min_i + 1 < 10):
            continue
        min_array.append(min_i + 3)
        max_array.append(max_i - 3)
        
        reduced_masks = masks[4:len(masks) - 3]
        reduced_volume = images[4:len(images) - 3]
        
        for i in reduced_masks:
            mask_array.append(i)
        count = 0
        for i in reduced_volume:
            used_patients.append(s[4:-4] + f"_{count}")
            volume_array.append(i)
            count += 1
        print(np.shape(volume_array))
        #print(max_i, min_i, max_i - min_i + 1)
        
        print(s[4:-4])
        total_images += max_i - min_i + 1 - 7

In [ ]:
import nibabel as nib
import numpy as np
import os
from scipy.ndimage import gaussian_filter
import cv2
from PIL import Image

directory = 'segmented-images-val'
volume_directory = 'AlzheimersFLAIR/data-preprocessing/n4-corrected2'

mask_array_val = []
volume_array_val = []


total_images = 0

used_patients_val = []

for s in os.listdir(directory):
    if (s != '.DS_Store'):
        filepath = os.path.join(directory, s)
        target_size = (224, 224)
        
        img = contrast_stretch(nib.load(filepath).get_fdata().transpose(2, 1, 0))
        #print(np.shape(img))
        min_i = 0
        max_i = 0
        
        volume_path = os.path.join(volume_directory, s[4:] + '.gz')
        volume = np.array([contrast_stretch(i) * 255 for i in nib.load(volume_path).get_fdata().transpose(2, 1, 0)]).astype(np.uint8)
        masks = []
        images = []
        for i in range(len(img)):
            segmented = np.sum(img[i]) > 0
            if (segmented):
                if (min_i == 0):
                    min_i = i
                max_i = i
                masks.append(cv2.resize(img[i], target_size, interpolation = cv2.INTER_LINEAR))
                images.append(cv2.resize(volume[i], target_size, interpolation = cv2.INTER_LINEAR))
                #print(i, np.sum(img[i]))
        if (max_i - min_i + 1 < 10):
            continue
        min_array.append(min_i + 3)
        max_array.append(max_i - 3)
        
        reduced_masks = masks[4:len(masks) - 3]
        reduced_volume = images[4:len(images) - 3]
        
        for i in reduced_masks:
            mask_array_val.append(i)
        count = 0
        for i in reduced_volume:
            used_patients_val.append(s[4:-4] + f"_{count}")
            volume_array_val.append(i)
            count += 1
        print(np.shape(volume_array_val))
        #print(max_i, min_i, max_i - min_i + 1)
        
        print(s[4:-4])
        total_images += max_i - min_i + 1 - 7

In [ ]:
import matplotlib.pyplot as plt
plt.figure()
plt.imshow(volume_array_val[0])
plt.imshow(mask_array_val[3], alpha = 0.43)
plt.show()

In [ ]:
from PIL import Image
Image.fromarray((mask_array[0] * 255).astype(np.uint8)).save("test.png")

In [ ]:
mask_array = np.repeat(np.expand_dims(mask_array, axis = -1), 3, axis = -1)
volume_array = np.repeat(np.expand_dims(volume_array, axis = -1), 3, axis = -1)
mask_array_val = np.repeat(np.expand_dims(mask_array_val, axis = -1), 3, axis = -1)
volume_array_val = np.repeat(np.expand_dims(volume_array_val, axis = -1), 3, axis = -1)

In [ ]:
print(np.shape(volume_array_val))

In [ ]:
data_gen_args = dict(
    rotation_range=30,
    zoom_range = 0.1,
    width_shift_range=0.1,
    height_shift_range=0.1)

In [ ]:
batch_size = 32
seed = 1
img_datagen = ImageDataGenerator(rescale = 1./255, **data_gen_args)
#img_datagen.fit(volume_array, augment = True, seed = seed)
img_generator = img_datagen.flow(
    volume_array,
    #target_size = (218, 182),
    #class_mode = None,
    seed = seed
)

mask_datagen = ImageDataGenerator(**data_gen_args)
#mask_datagen.fit(mask_array, augment = True, seed = seed)
mask_generator = mask_datagen.flow(
    #resize = (256, 256),
    mask_array,
    #class_mode = None,
    seed = seed
)

img_datagen_val = ImageDataGenerator(rescale = 1./255)
img_generator_val = img_datagen_val.flow(
    volume_array_val,
    seed = seed
)
mask_datagen_val = ImageDataGenerator()
mask_generator_val = mask_datagen_val.flow(
    mask_array_val,
    seed = seed
)

train_generator = zip(img_generator, mask_generator)
val_generator = zip(img_generator_val, mask_generator_val)

In [ ]:
img, mask = next(train_generator)
mask[mask > 0] = 1
#print(np.unique(x.astype(np.uint8)))
plt.figure()
plt.imshow(img[0])
plt.imshow(mask[0], alpha = 0.3)
plt.show()

In [ ]:
# https://lars76.github.io/neural-networks/object-detection/losses-for-segmentation/
#Can only beused for binary purposes
def dice_loss(y_true, y_pred):
    numerator = 2 * tf.reduce_sum(y_true * y_pred)
    # some implementations don't square y_pred
    denominator = tf.reduce_sum(y_true + tf.square(y_pred))

    return 1-(numerator / (denominator + tf.keras.backend.epsilon()))

In [ ]:
input_layer = Input((224, 224, 3))
c1 = Conv2D(16, 3, activation = "relu", kernel_initializer = 'he_normal', 
            name = 'c1_0', padding = "same")(input_layer)
d1 = Dropout(0.0, name = 'd1')(c1)
c1 = Conv2D(16, 3, activation = "relu", kernel_initializer = 'he_normal', name = 'c1_1', padding = "same")(d1)
p1 = MaxPooling2D(pool_size = (2, 2), strides = (2, 2), name = 'p1')(c1)

c2 = Conv2D(64, 3, activation = "relu", kernel_initializer = 'he_normal', name = 'c2_0', padding = "same")(p1)
d2 = Dropout(0.0, name = 'd2')(c2)
c2 = Conv2D(64, 3, activation = "relu", kernel_initializer = 'he_normal', name = 'c2_1', padding = "same")(c2)
p2 = MaxPooling2D(pool_size = (2, 2), strides = (2, 2), name = 'p2')(c2)

c3 = Conv2D(128, 3, activation = "relu", kernel_initializer = 'he_normal', name = 'c3_0', padding = "same")(p2)
d3 = Dropout(0.0, name = 'd3')(c3)
c3 = Conv2D(128, 3, activation = "relu", kernel_initializer = 'he_normal', name = 'c3_1', padding = "same")(d3)
p3 = MaxPooling2D(pool_size = (2, 2), strides = (2, 2), name = 'p3')(c3)

c4 = Conv2D(256, 3, activation = "relu", kernel_initializer = 'he_normal', name = 'c4_0', padding = "same")(p3)
d4 = Dropout(0.0, name = 'd4')(c4)
c4 = Conv2D(256, 3, activation = "relu", kernel_initializer = 'he_normal', name = 'c4_1', padding = "same")(d4)
p4 = MaxPooling2D(pool_size = (2, 2), strides = (2, 2), name = 'p4')(c4)

c5 = Conv2D(512, 3, activation = "relu", kernel_initializer = 'he_normal', name = 'c5_0', padding = "same")(p4)
c5 = Conv2D(512, 3, activation = "relu", kernel_initializer = 'he_normal', name = 'c5_1', padding = "same")(c5)
d5 = Dropout(0.0, name = 'd5')(c5)

u6 = Conv2DTranspose(256, (2, 2), (2, 2), activation = "relu", kernel_initializer = 'he_normal', 
                     name = 'u6', padding = "same")(d5)
m6 = concatenate([c4, u6], axis = 3)
c6 = Conv2D(256, 3, activation = "relu", kernel_initializer = 'he_normal', name = 'c6_0', padding = "same")(m6)
d6 = Dropout(0.0, name = 'd6')(c6)
c6 = Conv2D(256, 3, activation = "relu", kernel_initializer = 'he_normal', name = 'c6_1', padding = "same")(d4)

u7 = Conv2DTranspose(128, (2, 2), (2, 2), activation = "relu", kernel_initializer = 'he_normal',
                    name = 'u7', padding = "same")(c6)
m7 = concatenate([c3, u7], axis = 3)
c7 = Conv2D(128, 3, activation = "relu", kernel_initializer = 'he_normal', name = 'c7_0', padding = "same")(m7)
d7 = Dropout(0.0, name = 'd7')(c7)
c7 = Conv2D(128, 3, activation = "relu", kernel_initializer = 'he_normal', name = 'c7_1', padding = "same")(d7)

u8 = Conv2DTranspose(64, (2, 2), (2, 2), activation = "relu", kernel_initializer = 'he_normal', 
                     name = 'u8', padding = "same")(c7)
m8 = concatenate([c2, u8], axis = 3)
c8 = Conv2D(64, 3, activation = "relu", kernel_initializer = 'he_normal', name = 'c8_0', padding = "same")(m8)
d8 = Dropout(0.0, name = 'd8')(c8)
c8 = Conv2D(64, 3, activation = "relu", kernel_initializer = 'he_normal', name = 'c8_1', padding = "same")(d8)

u9 = Conv2DTranspose(16, (2, 2), (2, 2), activation = "relu", kernel_initializer = 'he_normal',
                    name = 'u9', padding = "same")(c8)
m9 = concatenate([c1, u9], axis = 3)
c9 = Conv2D(16, 3, activation = "relu", kernel_initializer = 'he_normal', name = 'c9_0', padding = "same")(m9)
d9 = Dropout(0.0, name = 'd9')(c9)
c9 = Conv2D(16, 3, activation = "relu", kernel_initializer = 'he_normal', name = 'c9_1', padding = "same")(d9)

out = Conv2D(1, 1, activation = "sigmoid")(c9)

model = Model(input_layer, out)
model.compile(optimizer = Adam(learning_rate = 0.001), 
             loss = dice_loss, metrics = ['accuracy'])

In [ ]:
model.summary()

In [ ]:
history = model.fit(
    train_generator,
    steps_per_epoch=len(mask_generator),
    validation_data=val_generator,
    validation_steps= len(mask_generator_val),
    epochs=75,
    verbose = 1
    )

In [ ]:
from keras.utils.vis_utils import plot_model
plot_model(model, to_file='modelUnetV2_plot.png', show_shapes=True, show_layer_names=True)

In [ ]:
model.save('Unet-results/UnetV2/UnetV2-end.h5')

In [ ]:
test = model.predict(img_generator)

In [ ]:
model.evaluate(test_generator) # still need to make test_generator

In [ ]:
print(history.history.keys())
# summarize history for accuracy
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()
# summarize history for loss
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

In [ ]:
plt.figure()
plt.imshow(test[19])
plt.show()

In [ ]:
np.unique(test[5])

In [ ]:
import nibabel as nib
import numpy as np
import os
from scipy.ndimage import gaussian_filter
import cv2
from PIL import Image

directory = 'segmented-images-test'
volume_directory = 'AlzheimersFLAIR/data-preprocessing/n4-corrected2'

mask_array_val = []
volume_array_val = []


total_images = 0

used_patients_val = []

for s in os.listdir(directory):
    if (s != '.DS_Store'):
        filepath = os.path.join(directory, s)
        target_size = (224, 224)
        
        img = contrast_stretch(nib.load(filepath).get_fdata().transpose(2, 1, 0))
        #print(np.shape(img))
        min_i = 0
        max_i = 0
        
        volume_path = os.path.join(volume_directory, s[4:] + '.gz')
        volume = np.array([contrast_stretch(i) * 255 for i in nib.load(volume_path).get_fdata().transpose(2, 1, 0)]).astype(np.uint8)
        masks = []
        images = []
        for i in range(len(img)):
            segmented = np.sum(img[i]) > 0
            if (segmented):
                if (min_i == 0):
                    min_i = i
                max_i = i
                masks.append(cv2.resize(img[i], target_size, interpolation = cv2.INTER_LINEAR))
                images.append(cv2.resize(volume[i], target_size, interpolation = cv2.INTER_LINEAR))
                #print(i, np.sum(img[i]))
        if (max_i - min_i + 1 < 10):
            continue
        min_array.append(min_i + 3)
        max_array.append(max_i - 3)
        
        reduced_masks = masks[4:len(masks) - 3]
        reduced_volume = images[4:len(images) - 3]
        
        for i in reduced_masks:
            mask_array_val.append(i)
        count = 0
        for i in reduced_volume:
            used_patients_val.append(s[4:-4] + f"_{count}")
            volume_array_val.append(i)
            count += 1
        print(np.shape(volume_array_val))
        #print(max_i, min_i, max_i - min_i + 1)
        
        print(s[4:-4])
        total_images += max_i - min_i + 1 - 7

In [ ]:
np.shape(volume_array_val)

In [ ]:
volume_array_val = np.repeat(np.expand_dims(volume_array_val, -1), 3, axis = -1)
mask_array_val = np.repeat(np.expand_dims(mask_array_val, -1), 3, axis = -1)

In [ ]:
batch_size = 32
seed = 1

img_datagen_test = ImageDataGenerator(rescale = 1./255)
img_generator_test = img_datagen_test.flow(
    volume_array_val,
    seed = seed
)
mask_datagen_test = ImageDataGenerator()
mask_generator_test = mask_datagen_test.flow(
    mask_array_val,
    seed = seed
)

#train_generator = zip(img_generator, mask_generator)
test_generator = zip(img_generator_test, mask_generator_test)

In [ ]:
model.evaluate(test_generator, steps = len(mask_generator_test))

In [ ]:
test_results = model.predict(test_generator, steps = len(mask_generator_test))

In [ ]:
fig = plt.figure(figsize=(60, 48))
rows, columns = 2, 25
count1 = 0
count2 = 0

test_x = []
test_y = []

test_pred = []

for i in range(len(mask_generator_test)):
    x, y = next(test_generator)
    pred = model.predict(x)
    
    for i in range(len(x)):
        test_x.append(x[i])
        test_y.append(y[i])
        test_pred.append(pred[i])


print('Predicted values:')
# for i in range(len(test_x)):
#     if (count1 > 14 and count1 <= 24):
#         #print(count)
#         fig.add_subplot(rows, columns, i - 7)
#         plt.imshow(test_x[i][:, :, 0])
#         plt.imshow(test_y[i], alpha = 0.23)
#         plt.axis('off')
#         plt.title(f"Image {i - 10}")
#     count1 += 2


for i in range(len(test_x)):
    if (count2 > 14 and count2 <= 24):
        fig.add_subplot(rows, columns, count1 + 1)
        plt.imshow(test_x[i][:, :, 0])
        plt.imshow(test_pred[i], alpha = 0.23)
        plt.axis('off')
        plt.title(f"Image {i - 7}")
        count1 += 1
    count2 += 2
#     for i in range(len(x)):
#         plt.figure()
#         plt.imshow(x[i])
#         plt.imshow(pred[i], alpha = 0.23)
#         plt.show()

In [ ]:
model.load_weights('AlzheimersFLAIR/LateralVentricleSegmentation/Unet-results/UnetV2/UnetV2-end.h5')

In [ ]:
directory = 'AlzheimersFLAIR/data-preprocessing/n4-corrected2'

out_directory = 'AlzheimersFLAIR/2D-models/largest-ventricles2'

problematic_registers = []
# patient ids removed for privacy purposes

selected_slices = []


for s in os.listdir(directory):
    if (s != '.DS_Store'):
        jpg_name = s[:-7] + '.jpg'
        print(jpg_name)
        if jpg_name in problematic_registers:
            print("found problematic register, skipping...")
            continue
        
        filepath = os.path.join(directory, s)
        img = np.array([contrast_stretch(i) * 255 for i in nib.load(filepath).get_fdata().transpose(2, 1, 0)]).astype(np.uint8)[12:28]
        img = np.array([cv2.resize(i, target_size, interpolation = cv2.INTER_LINEAR) for i in img])
        img = np.repeat(np.expand_dims(img, -1), 3, axis = -1)
        
        out_array = contrast_stretch(nib.load(filepath).get_fdata().transpose(2, 0, 1))[12:28]
        
        print(np.shape(img))
        masks = model.predict(img)
        print(np.shape(masks))
        
        largest_index = 0
        for i in range(len(masks)):
            current_area = np.sum(masks[i])
            largest_area = np.sum(masks[largest_index])
            if (current_area > largest_area):
                largest_index = i
        
        jpg_path = out_directory
        patient_id = s[:10]
        # print(patient_id)  # removed for privacy
        diagnosis = labels.loc[labels['Subject ID'] == patient_id, 'Research Group'].values[0]
        if (diagnosis == 'MCI'): continue
        if (diagnosis == 'EMCI'):
            jpg_path = os.path.join(jpg_path, '1')
        elif (diagnosis == 'LMCI'):
            jpg_path = os.path.join(jpg_path, '2')
        elif (diagnosis == 'AD'):
            jpg_path = os.path.join(jpg_path, '3')
        else:
            jpg_path = os.path.join(jpg_path, '0')
        jpg_path = os.path.join(jpg_path, jpg_name)
        out_mask = Image.fromarray(np.rot90((out_array[largest_index - 5] * 255).astype(np.uint8), k = 1))
        out_mask.save(jpg_path)
        